In [1]:
from sklearn.datasets import fetch_california_housing

# Yeh dono tools humare data ko bar-bar split karne aur check karne ke kaam aate hain (Cross-Validation ke liye).
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression   # THIS will prediict Houses prices 
import numpy as np  # Numpy for maths and calculations

In [2]:

# Load built-in housing dataset
# X (Features): Isme gharon ki details hain (jaise average rooms, area ki income, house age wagera).
# Isko dekh kar model seekhta hai.
# y (Target/Label): Isme asal qeematen (House Prices) hain, jo model ko predict karni hain.

housing = fetch_california_housing()
X, y = housing.data, housing.target

In [3]:
# Single split (old way)
from sklearn.model_selection import train_test_split  

# Humne poore data ko 80% training (X_train, y_train) aur 20% testing (X_test, y_test)
# mein divide kar diya.

# random_state=42 ka matlab hai ke data hamesha ek hi tarah se random split ho (taake har dafa run karne par same result aaye).
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()

# Humne LinearRegression ka model banaya.
# model.fit ka matlab hai ke model ne 80% training data ko dekh kar gharon ki qeematen predict
# karna seekh (train) liya hai.
model.fit(X_train, y_train)

# model.score se hum check kar rahe hain ke bache hue 20% testing data par model kitna acha perform kar raha hai.
# Yeh humein $R^2$ Score deta hai (jo batata hai ke model ki prediction qeematen asal qeematon ke kitne kareeb hain).
single_r2 = model.score(X_test, y_test)

In [4]:
# Cross-validation (professional way)
# Single Split ka masla kya hai? Agar humne jo 20% test data alag kiya, kismat se woh bohot aasan data nikal aaya, 
# to model ki accuracy bohot zyada dikhayi degi. Agar woh mushkil data hua, to accuracy bohot kam dikhayi degi. Is
# masle ko hal karne ke liye hum K-Fold Cross-Validation use karte hain.

# KFold(n_splits=5): Iska matlab hai hum pooray data ke 5 barabar hissay (Folds) karenge.

# shuffle=True: Data ko pehle achi tarah mix (shuffled) kiya jata hai taake har hissay mein har tarah ka data aaye.

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Round 1: Pehle 4 hisson par model ko train karega, aur 5th hissay par test karega. Ek $R^2$ score save kar lega.
# Round 2: Ab kisi aur hissay ko test data banayega aur baqi 4 par train karega. Dusra $R^2$ score save karega.
# Round 3, 4, 5: Isi tarah har ek hissay ko baari-baari test data bana kar total 5 scores nikalega.
cv_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')


In [5]:
# Yeh aapko single split (woh 80-20 wala) ka $R^2$ score dikhayega.
print(f"Single split R²:  {single_r2:.4f}")

# cv_scores.mean(): Jo 5 alag-alag scores aaye thay, yeh unka Average (Mean) nikalta hai. Yeh score sabse zyada 
# bharosemand hota hai kyunki yeh pure data par test hone ke baad aaya hai.
print(f"CV mean R²:       {cv_scores.mean():.4f}")

# cv_scores.std(): Standard Deviation batata hai ke humare 5 scores aapas mein kitne farq par hain. Agar yeh bohot
# kam hai (jaise 0.01 ya 0.02), iska matlab hai model har tarah ke data par stable perform kar raha hai.
print(f"CV std R²:        {cv_scores.std():.4f}")

# Yeh aapko un paanchon ($5$) rounds ke alag-alag scores list ki soorat mein dikha deta hai.
print(f"All fold scores:  {cv_scores.round(4)}")

Single split R²:  0.5758
CV mean R²:       0.6014
CV std R²:        0.0170
All fold scores:  [0.5758 0.6137 0.6086 0.6213 0.5875]


In [6]:
# NOW MOVE TO NEXT TASKS         NOW MOVE TO PIPELINE 

# Machine learning projects mein aksar hum data ko transform (clean/scale) karna bhool jaate hain ya
# train aur test data mix ho jata hai (Data Leakage). Pipeline is masle ko hal karti hai.

In [7]:
# Pipeline: Yeh ek factory assembly line ki tarah hai. Aap isko steps de dete hain (jaise: pehle data saaf karo,
# phir model train karo), aur yeh saare kaam ek sequence mein khud ba khud kar deti hai.
from sklearn.pipeline import Pipeline

# ColumnTransformer: Hamare dataset mein alag-alag columns hote hain (kuch numbers, kuch text). Yeh tool madad 
# karta hai ke hum different columns par different transformations (operations) apply kar sakein.
from sklearn.compose import ColumnTransformer

# StandardScaler: Yeh numbers ko scale karta hai (unka mean 0 aur standard deviation 1 kar deta hai) taake bade 
# numbers (jaise salary ya ghar ki qeemat) model ko confuse na karein.

# OneHotEncoder: Yeh text columns (jaise 'City' ya 'Yes/No') ko numbers (0 aur 1) mein convert karta hai 
# (halaanki aapke current code mein yeh aage use nahi ho raha, par import kiya hua hai).
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# RandomForestRegressor: Yeh ek advanced aur powerful Machine Learning model hai jo ek decision tree ki jagah
# bohot saare "Trees" (trees ka forest) bana kar prediction karta hai.
from sklearn.ensemble import RandomForestRegressor
import pandas as pd


In [8]:
# Separate column types
# X_df: Yeh ek variable ka naam hai jahan hum apna saaf suthra data (table ki shakal mein) store kar rahe hain.
# DataFrame(): Yeh Pandas ka ek function hai jo raw numbers/array ko ek proper Row and Column wale table (sheet) 
# mein convert karta hai.

# housing.data: Isme California housing dataset ka raw data (bina column names ke) para hua tha.

# columns=housing.feature_names: Iska matlab hai ke "Table ke columns ko unke asli naam de do" (jaise MedInc,
# HouseAge wagera), taake humein pata chale kaun sa column kis cheez ka hai.
X_df = pd.DataFrame(housing.data, columns=housing.feature_names)



# num_cols: Yeh ek variable ka naam hai jahan hum sirf numbers wale columns ke names save karenge.
# X_df.select_dtypes(...): Yeh function pooray table mein se columns ko unki type ke hissab se filter karta hai.
# include='number': Hum keh rahe hain ke "Sirf woh columns select karo jin mein numbers (integers ya floats) hain."
# (Agar koi text ya category wala column hoga, to woh chhut jayega).
# .columns: Yeh un selected columns ke names nikaal leta hai.
# .tolist(): Yeh un names ko ek simple Python list (jaise ['MedInc', 'HouseAge', ...] ) mein convert kar 
# deta hai taake hum aage use kar sakein.

num_cols = X_df.select_dtypes(include='number').columns.tolist()


In [9]:
# Preprocessor
# preprocessor: Yeh hamari data saaf/scale karne wali machine ka naam hai.
# ColumnTransformer(...): Scikit-Learn ka ek tool hai jo batata hai ke dataset 
# ke alag-alag columns par alag-alag operations kaise karne hain.
# transformers=[...]: Isme hum rules ki ek list dete hain

# StandardScaler(): Yeh asal operation (transformer) hai. Iska kaam hai data ko 
# scale karna (taake mean 0 ho jaye aur variance 1). Yeh zaroori hai kyunki kuch
# columns mein bohot bade numbers hote hain aur kuch mein bohot chote, scale 
# karne se model sahi seekhta hai.

# num_cols: Humne isay bataya ke yeh StandardScaler sirf un columns par chalana jo humne upar num_cols list mein save kiye hain.
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols)
])


In [10]:
# Full pipeline: preprocessing + model

# pipe: Yeh hamari poori Automatic Assembly Line ka naam hai.

# Pipeline([...]): Yeh Scikit-Learn ka tool hai jo multi-step process ko ek chain (kadi) mein baandh deta hai.
# Isme data khud-ba-khud ek step se doosre step mein jata hai.

# ('preprocessor', preprocessor):
# Pehla step (Step 1): Iska naam humne 'preprocessor' rakha, aur isme humne upar banaya hua preprocessor
# dal diya jo data ko scale karega.

# ('model', RandomForestRegressor(...)):
# Doosra step (Step 2): Iska naam humne 'model' rakha. Jab data pehle step se scale ho kar niklega, toh 
# woh is model ke paas jayega.

# RandomForestRegressor: Yeh hamara machine learning model hai (jo bohot saare decision trees ko mila kar
# ghar ki price predict karta hai).

# n_estimators=100: Iska matlab hai hum model ko keh rahe hain ke prediction ke liye 100 alag-alag decision 
# trees banao aur unka average lo (is se accuracy achi aati hai).

# random_state=42: Yeh randomness ko control karta hai taake jab bhi aap code run karein, aapka model hamesha
# same result de.


pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])


In [11]:
# One line — pipeline handles everything

# cv_scores: Yeh ek variable hai jahan hamare paanchon ($5$) rounds ke test scores save honge.

# cross_val_score(...): Yeh function automated testing karta hai (K-Fold Cross-Validation).
#     Yeh pooray kaam ko aasan bana deta hai.\

# pipe: Humne isay apni banayi hui Pipeline de di. Iska sabse bada faida yeh hai ke cross_val_score
# jab bhi data ko train aur test folds mein divide karega, woh sirf training fold par scaler ko fit 
# karega aur test fold ko bilkul touch nahi karega (is se Data Leakage ka khatra khatam ho jata hai).

# X_df: Yeh hamara input features data (table) hai.

# y: Yeh hamara target data (asli prices) hai jo model ko predict karna seekhna hai.

# cv=5: Iska matlab hai 5-Fold Cross Validation. Data ke 5 hissay honge aur testing 5 rounds mein hogi 

# scoring='r2': Humne isay bataya ke model ki performance check karne ka criteria $R^2$ Score 
# (Coefficient of Determination) hona chahiye (jo 0 aur 1 ke beech hota hai; 1 ke jitna kareeb ho, model utna hi behtar hai).
# (har round mein ek alag hissa test data banega).

cv_scores = cross_val_score(pipe, X_df, y, cv=5, scoring='r2')


print(f"Pipeline CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

Pipeline CV R²: 0.6559 ± 0.0785


In [12]:
# TASK 3 HYPER PARAMETER  

In [13]:
# sklearn (Scikit-Learn) Python ki sabse badi machine learning library hai.
# model_selection uski ek branch hai jisme models ko test aur tune karne wale tools hote hain.

# GridSearchCV (Grid Search Cross-Validation) ek aisa automatic tool hai jo humare bataye gaye 
# settings (parameters) ke har combination ko khud ba khud test karke sabse best settings nikaal kar deta hai
from sklearn.model_selection import GridSearchCV
# 1. Classifier ki jagah Regressor import karein

# ensemble ka matlab hota hai "group" ya "ek sath mil kar kaam karna".

# RandomForestRegressor hamara machine learning model hai. Yeh ek decision tree ke bajaye bohot saare 
# (jaise 100 ya 200) decision trees ka ek "forest" (jungle) banata hai aur un sab ki predictions ka 
# average le kar final price batata hai. Is se accuracy bohat achi aati hai.


from sklearn.ensemble import RandomForestRegressor

# pandas data ko tables (rows and columns) ki shakal mein analyze karne ke liye use hoti hai. Humne isko chota naam pd de diya hai.
import pandas as pd

In [14]:
# Parameter grid wahi rahega

# param_grid: Yeh ek dictionary (options ki list) hai jisme hum RandomForest ke parameters ki settings 
# de rahe hain taake grid search inhein baari-baari test kare:

# n_estimators: Model mein kitne decision trees hone chahiye? Hum check karna chahte hain ke $50$, $100$
# ya $200$ trees mein se kis par best result milta hai.

# max_depth: Har tree kitna lamba/gehra ho sakta hai? Options hain: $3$, $5$, $10$ ya None (unlimited gehrai).
# min_samples_split: Ek branch ko mazeed do hisson mein todne ke liye kam az kam kitne samples hone chahiye?
# Options hain: $2$ ya $5$.
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5]
}

In [15]:
# 2. GridSearchCV ko Regressor aur 'r2' scoring ke sath setup karein

# grid_search: Yeh hamari automatic tuning machine ka naam hai.

# RandomForestRegressor(random_state=42): Humne isay bataya ke tumne is model ko 
# tune karna hai. random_state=42 isliye likha taake har dafa run karne par
# same trees banein aur result badle nahi

# param_grid: Humne apni settings ki list is machine ko de di.

# cv=5: Iska matlab hai 5-Fold Cross Validation. Data ke 5 hissay honge. Har combination
# ko $5$ dafa test kiya jayega.Total Fits: $24 \text{ combinations} \times 5 \text{ folds} = 120$
# fits (yaani total 120 dafa model train hoga).

# scoring='r2': Model ko judge karne ka criteria $R^2$ Score hai. Yeh continuous values (jaise house prices)
# ke liye use hota hai. 1 ke jitna kareeb ho, model utna hi behtar hota hai.

# n_jobs=-1: Computer ke saare processors ko ek sath kaam par laga do taake calculation jaldi ho jaye.
# verbose=1: Screen par progress messages show karo (jaise "Fitting 5 folds...").


grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),  # <-- CHANGE HERE: Regressor
    param_grid,
    cv=5,
    scoring='r2',                            # <-- CHANGE HERE: 'r2' scoring
    n_jobs=-1,
    verbose=1
)

# 3. Model ko fit (train) karein
# grid_search.fit: Is line par machine chalna shuru hoti hai. Yeh X_train (features) aur y_train (asli prices) 
# ko le kar saare $120$ combinations ko baari-baari train aur test karegi aur sabse best model ko save kar legi.
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


,estimator,RandomForestR...ndom_state=42)
,param_grid,"{'max_depth': [3, 5, ...], 'min_samples_split': [2, 5], 'n_estimators': [50, 100, ...]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [19]:
import numpy as np

# 1. Check karein ke kya array mein koi NaN (missing value) hai?
# Agar data mein koi khali jagah (NaN) ho, toh model crash kar jata hai. Isliye hum check kar rahe hain:

# np.isnan(X_train): Yeh pooray array mein har jagah check karta hai ke kahin missing value to nahi hai.

# .any(): Agar pooray data mein ek bhi missing value hui, toh yeh True dega, nahi toh False.

#

has_nan = np.isnan(X_train).any()
print("Kya data mein missing values (NaN) hain?:", has_nan)

# 2. Agar hain, to total kitni missing values hain?

 # if has_nan: Agar missing values hain, toh .sum() unki total count (tadad) print kar dega taake humein
# pata chale kitna data missing hai.
if has_nan:
    print("Total missing values:", np.isnan(X_train).sum())

Kya data mein missing values (NaN) hain?: False


In [20]:
# 1. Pehle target variable ki type check karein

# y_train[:5]: Hum target variable (yaani jo cheez predict karni hai) ki pehli 5 values print karke confirm 
# kar rahe hain ke kya yeh sach mein continuous prices (numbers) hain ya koi categorical labels (Yes/No).
# Is se classification aur regression ka farq saaf ho jata hai.


print("y_train ki pehli 5 values:", y_train[:5])

# 2. Agar yeh continuous numbers (decimals/prices) hain, to Classifier ki jagah Regressor use karein:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

# RandomForestClassifier ki jagah RandomForestRegressor likhein
grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),  # <-- CHANGE HERE
    param_grid,
    cv=5,
    scoring='r2',                            # Regressor ke liye 'r2' ya 'neg_mean_squared_error' scoring hoti hai
    n_jobs=-1,
    verbose=1
)

# Ab isay fit karein
grid_search.fit(X_train, y_train)

y_train ki pehli 5 values: [1.03  3.821 1.726 0.934 0.965]
Fitting 5 folds for each of 24 candidates, totalling 120 fits


,estimator,RandomForestR...ndom_state=42)
,param_grid,"{'max_depth': [3, 5, ...], 'min_samples_split': [2, 5], 'n_estimators': [50, 100, ...]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [21]:

# grid_search.best_params_: GridSearchCV ke saare combinations chalne ke baad, jis setting par sabse
# zyada $R^2$ score aaya,
# yeh us behtareen setting ko print kar deta hai.

print(f"Best params: {grid_search.best_params_}")

# grid_search.best_score_: Yeh us sabse behtareen setting ka average cross-validation $R^2$ score print karta hai
# (jaise decimal ke baad 4 digits tak: :.4f).

print(f"Best CV AUC: {grid_search.best_score_:.4f}")

# Top 5 combinations

# grid_search.cv_results_: Isme saare combinations ke bohot saare detailed results hote hain. Humne unhein Pandas
# ke tabular format (DataFrame) mein convert kar diya taake dekhna aasan ho.

results = pd.DataFrame(grid_search.cv_results_)

# Humne pooray detailed table mein se sirf 3 kaam ke columns chun liye:
# 'params': Jo settings humne use ki thin.
# 'mean_test_score': Un settings par milne wala average score.
# 'rank_test_score': Unka ranking number (1st, 2nd, 3rd...).   


top5 = results[['params','mean_test_score','rank_test_score']]

# sort_values('rank_test_score'): Pure table ko rank ke hisab se tarteeb diya (Rank 1 sabse upar aa jaye).
print(top5.sort_values('rank_test_score').head(5))

Best params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Best CV AUC: 0.8049
                                               params  mean_test_score  \
20  {'max_depth': None, 'min_samples_split': 2, 'n...         0.804875   
23  {'max_depth': None, 'min_samples_split': 5, 'n...         0.804662   
22  {'max_depth': None, 'min_samples_split': 5, 'n...         0.804225   
19  {'max_depth': None, 'min_samples_split': 2, 'n...         0.804153   
21  {'max_depth': None, 'min_samples_split': 5, 'n...         0.802240   

    rank_test_score  
20                1  
23                2  
22                3  
19                4  
21                5  


In [22]:
#NOW TASK NUMBER 4 COMES

In [2]:
# Complete professional ML workflow
from sklearn.datasets import fetch_california_housing
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np
import pandas as pd

housing = fetch_california_housing()
X, y = housing.data, housing.target

# Professional pipeline

# ('scaler', StandardScaler()): Pehla kaam hoga data ko scale karna.
# ('rf', RandomForestRegressor(...)): Scale hone ke baad data seedha is model ke paas seekhne ke liye jayega.


pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(random_state=42))
])

# Hyperparameter tuning


# param_grid: Humne grid search ko tuning ke liye options diye.

# rf__ (Double Underscore): Yeh pipeline ka rule hai! Iska matlab hai ke pipeline ke andar jo 'rf' naam ka
# step hai, uske andar n_estimators aur max_depth ko change karo.

# Total combinations: $2 \text{ (trees)} \times 3 \text{ (depths)} = 6$ combinations. Aur cv=5 hai, 
# toh total fits honge $6 \times 5 = 30$ fits.

param_grid = {'rf__n_estimators': [100, 200], 'rf__max_depth': [10, 20, None]}

# Humne GridSearchCV machine chalayi jo 5-Fold Cross Validation ke sath saare combinations check karegi
# aur grid.fit(X, y) par background mein 30 dafa model ko train karegi.


grid = GridSearchCV(pipe, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid.fit(X, y)


# grid.best_params_: Sabse best setting print karega (Aapke case mein: 100 trees aur None depth).grid.best_score_: Us 
# best setting ka average $R^2$ score print karega (Aapka score aaya: 0.6559).
print(f"Best params: {grid.best_params_}")
print(f"Best CV R²: {grid.best_score_:.4f}")

# Feature importance

# grid.best_estimator_: Iska matlab hai "Woh poori pipeline uthao jo sabse best perform kar rahi thi".

# .named_steps['rf']: Poori pipeline mein se humne 'scaler' ko chora aur sirf uske andar maujood train
# hue RandomForestRegressor model ko nikaal kar best_rf variable mein save kar liya.

best_rf = grid.best_estimator_.named_steps['rf']

# best_rf.feature_importances_: RandomForest model training ke baad khud batata hai ke usne kis feature ko kitna 
# important samjha (Yeh decimal numbers ki ek list hoti hai).

# pd.Series(..., index=housing.feature_names): Humne un decimals ko Pandas ki ek Series (ek column wala table) banaya 
# aur uske aage un columns ke asli naam (MedInc, HouseAge wagera) set kar diye.

# .sort_values(ascending=False): Humne data ko sort kiya taake jo feature sabse zyada important hai (Highest percentage) 
# woh sabse upar aaye, aur kam important niche chale jayein.


feat_imp = pd.Series(best_rf.feature_importances_,
                     index=housing.feature_names).sort_values(ascending=False)

# Jab aap isay print karenge, toh aapke samne saare features ki Importance Percentage aa jayegi, jis se aapko pata chalega 
# ke California mein gharon ki qeemat badhne ya kam hone mein sabse bada haath kis column ka tha!

print(feat_imp)

Best params: {'rf__max_depth': None, 'rf__n_estimators': 100}
Best CV R²: 0.6559
MedInc        0.520036
AveOccup      0.136403
Latitude      0.092852
Longitude     0.092696
HouseAge      0.052963
AveRooms      0.044515
Population    0.031230
AveBedrms     0.029305
dtype: float64
